# 第13回 ゲーム探索：値とコードを一段ずつ読む

合法手を作り、葉を調べ、相手の選択を考えて値を戻します。省略の理由を実際の探索履歴で確認します。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** 固定木は値3・手A。ニム(3,2)はMAXの勝ち。αβも根の値3を保つ。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
from copy import deepcopy


In [ ]:
import json


In [ ]:
TREE_BASE = {"leaves": [3, 5, 2, 9, 0, 4], "heuristics": [0, 1, 4, 2],
             "depth": 2, "reverse": False, "algorithm": "minimax", "player": 1}


In [ ]:
NIM_BASE = {"piles": [3, 2], "player": 1, "reverse": False, "algorithm": "minimax"}


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 石を取った次の局面

**今いる段階：** 局面→合法手→次局面

**予想：** どこから何個取れるか

**操作：** example_pilesを(3, 2)から(2, 2)へ変更し、同じセルを実行します。

**結果の目印：** (3,2)は5合法手、(2,2)は4合法手

**戻す：** example_piles=(3, 2)に戻して再実行します。

**説明する：** 元の局面をコピーする理由


### 計算の定義：`integer`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def integer(value, low, high, name):
    if type(value) is not int or not low <= value <= high:
        raise ValueError(f"{name}: {low}〜{high}の整数を指定してください")
    return value


### 計算の定義：`validate_piles`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def validate_piles(piles):
    if not isinstance(piles, (list, tuple)) or len(piles) != 2:
        raise ValueError("山は2個です")
    for p in piles:
        integer(p, 0, 4, "石の数")
    if sum(piles) > 6:
        raise ValueError("石は合計6個以下です")


### 計算の定義：`legal_moves`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def legal_moves(piles):
    validate_piles(piles)
    moves = []
    for pile_index, stones in enumerate(piles):
        for take in range(1, stones + 1):
            moves.append([pile_index, take])
    return moves


### 計算の定義：`apply_move`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def apply_move(piles, action):
    validate_piles(piles)
    if not isinstance(action, (list, tuple)) or len(action) != 2:
        raise ValueError("手は[山番号,取る個数]です")
    i, take = action
    integer(i, 0, 1, "山番号")
    integer(take, 1, 4, "取る個数")
    if take > piles[i]:
        raise ValueError("その山には指定した数の石がありません")
    result = list(piles)
    result[i] -= take
    return result


### 計算の定義：`_search`

深さ優先探索。履歴中の±無限大はJSONで明示できる文字列にする。


In [ ]:
def _search(c, kind):
    """深さ優先探索。履歴中の±無限大はJSONで明示できる文字列にする。"""
    trace, pruned = [], []
    stats = {"visited_nodes": 0, "evaluations": 0, "terminal_evaluations": 0,
             "cutoff_evaluations": 0}
    ab = c["algorithm"] == "alphabeta"

    def bound_number(x):
        return "-inf" if x == -float("inf") else "+inf" if x == float("inf") else x

    def visit(state, player, depth, node_id, alpha, beta):
        stats["visited_nodes"] += 1
        trace.append({"event": "enter", "id": node_id, "player": player, "depth": depth,
                      "alpha": bound_number(alpha), "beta": bound_number(beta)})
        terminal = len(state) == 2 if kind == "tree" else sum(state) == 0
        cutoff = kind == "tree" and not terminal and depth >= c["depth"]
        if terminal or cutoff:
            if kind == "nim":
                value = -player  # 手番側は合法手がないので負け。
            elif terminal:
                value = c["leaves"]["ABC".index(state[0]) * 2 + int(state[1]) - 1]
            else:
                value = c["heuristics"][0 if state == "R" else "ABC".index(state) + 1]
            category = "terminal" if terminal else "cutoff"
            stats["evaluations"] += 1
            stats[category + "_evaluations"] += 1
            trace.append({"event": "evaluate", "id": node_id, "kind": category, "value": value})
            return value, None

        if kind == "tree":
            children = [(x, x, x) for x in (list("ABC") if state == "R" else [state + "1", state + "2"])]
        else:
            children = [(a, apply_move(state, a), f"{a[0]}:{a[1]}") for a in legal_moves(state)]
        if c["reverse"]:
            children.reverse()
        initial_alpha, initial_beta = alpha, beta
        value = -float("inf") if player == 1 else float("inf")
        best_action = None
        for index, (action, child, token) in enumerate(children):
            score, _ = visit(child, -player, depth + 1, node_id + "/" + token, alpha, beta)
            # 厳密な不等号で、同点時は最初の手を保持する。
            if best_action is None or (score > value if player == 1 else score < value):
                value, best_action = score, action
            if ab:
                if player == 1:
                    alpha = max(alpha, value)
                else:
                    beta = min(beta, value)
            trace.append({"event": "update", "id": node_id, "action": action, "value": value,
                          "alpha": bound_number(alpha), "beta": bound_number(beta)})
            if ab and alpha >= beta:
                skipped = [node_id + "/" + x[2] for x in children[index + 1:]]
                if skipped:
                    pruned.extend(skipped)
                    trace.append({"event": "prune", "id": node_id, "skipped": skipped,
                                  "value": value, "bound": "lower" if player == 1 else "upper"})
                break
        certainty = "upper" if ab and value <= initial_alpha else "lower" if ab and value >= initial_beta else "exact"
        trace.append({"event": "return", "id": node_id, "value": value, "bound": certainty})
        return value, best_action

    value, action = visit("R" if kind == "tree" else c["piles"], c["player"], 0, "R", -float("inf"), float("inf"))
    return {"value": value, "best_action": action, **stats, "pruned": pruned, "trace": trace}


### 計算の定義：`_config`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def _config(config, kind):
    base = TREE_BASE if kind == "tree" else NIM_BASE
    if not isinstance(config, dict) or set(config) - set(base):
        raise ValueError("未知の設定項目です")
    c = deepcopy({**base, **config})
    if type(c["player"]) is not int or c["player"] not in (-1, 1):
        raise ValueError("手番はMAX=1またはMIN=-1です")
    if type(c["reverse"]) is not bool or c["algorithm"] not in ("minimax", "alphabeta"):
        raise ValueError("探索順・方式が不正です")
    if kind == "tree":
        integer(c["depth"], 0, 2, "探索深さ")
        for key, length in (("leaves", 6), ("heuristics", 4)):
            if not isinstance(c[key], (list, tuple)) or len(c[key]) != length:
                raise ValueError(f"{key}は{length}個です")
            for x in c[key]:
                integer(x, -100, 100, key)
    else:
        validate_piles(c["piles"])
    return c


### 計算の定義：`search_nim`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def search_nim(config=None):
    return _search(_config({} if config is None else config, "nim"), "nim")


### 計算の定義：`summary`

基準JSONには値と計数を保存。完全な探索履歴は各result['trace']に残す。


In [ ]:
def summary(result):
    """基準JSONには値と計数を保存。完全な探索履歴は各result['trace']に残す。"""
    return {key: deepcopy(value) for key, value in result.items() if key != "trace"}


### 計算の定義：`lesson_moves`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_moves(piles=(3,2)):
    rows = []
    for action in legal_moves(piles):
        after = apply_move(piles,action)
        rows.append((list(piles),action[0],action[1],after))
    result = search_nim({'piles':list(piles)})
    return {'title':'局面から合法手を作る','question':'1回に両方の山から取れるか',
            'input':list(piles),'columns':['元の山','選ぶ山番号','取る個数','次の局面'],'rows':rows,
            'result':summary(result),'comparison':'(3,2)→(2,2)にすると合法手と勝敗を比べられる。',
            'explanation':'一つの山から1個以上取る。山番号は0始まり。利得はMAX側から見て勝ち+1・負け−1。探索は学習ではない。',
            'reset':'piles=(3,2)に戻す。元の局面を変更せず探索する。'}


In [ ]:
example_piles = (3, 2)
program_observation_moves = lesson_moves(example_piles)
learning_show(program_observation_moves, reset='example_piles=(3, 2)に戻して再実行します。')


## 2. 葉の値を戻す順序

**今いる段階：** 葉→MIN→MAX

**予想：** 葉9がある枝を必ず選ぶか

**操作：** example_b1を2から6へ変更し、同じセルを実行します。

**結果の目印：** 基準のMIN値A3・B2・C0からMAXはA3を選ぶ

**戻す：** example_b1=2に戻して再実行します。

**説明する：** MINは相手の立場で選ぶ


### 計算の定義：`search_tree`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def search_tree(config=None):
    return _search(_config({} if config is None else config, "tree"), "tree")


### 計算の定義：`lesson_minimax`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_minimax(leaf_b1=2):
    config = deepcopy(TREE_BASE)
    config['leaves'][2] = leaf_b1
    result = search_tree(config)
    rows = []
    for event in result['trace']:
        if event['event'] in ('evaluate','update','return'):
            rows.append((event['event'],event['id'],event.get('action','—'),event['value'],event.get('bound','—')))
    return {'title':'葉からMIN、MINからMAXへ値を戻す','question':'最も大きい葉9の枝を選べばよいか',
            'input':config['leaves'],'columns':['処理','節点','調べた子','現在値','値の確かさ'],'rows':rows,
            'tables':[('木の親子関係',['親・役割','子1','子2'],[(name+' MIN',config['leaves'][2*i],config['leaves'][2*i+1]) for i,name in enumerate('ABC')])],
            'result':summary(result),'comparison':'B1を2→6にすると、BでMINが返す値は2→6、根の選択もA→Bになる。',
            'explanation':'根RはMAX、A・B・CはMIN。相手が小さい値を選ぶことを織り込む。updateは調査済みの子だけの暫定値、returnはその探索の返却値。',
            'reset':'leaf_b1=2で再実行。'}


In [ ]:
example_b1 = 2
program_observation_minimax = lesson_minimax(example_b1)
learning_show(program_observation_minimax, reset='example_b1=2に戻して再実行します。')


## 3. どこまで読むか

**今いる段階：** 打切り位置→評価→選択

**予想：** 途中の評価と終端の利得は同じか

**操作：** example_depthを2から1へ変更し、同じセルを実行します。

**結果の目印：** 深さ1では推定値4・手B

**戻す：** example_depth=2に戻して再実行します。

**説明する：** 打切り評価の限界


### 計算の定義：`lesson_depth`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_depth(depth=2):
    result = search_tree({'depth':depth})
    rows = [(e['id'],e['kind'],e['value']) for e in result['trace'] if e['event']=='evaluate']
    return {'title':'終端の利得と途中の評価値を区別する','question':'浅く読むと同じ手を選ぶか',
            'input':f'深さ={depth}、途中評価R,A,B,C={TREE_BASE["heuristics"]}',
            'columns':['評価する節点','terminal=終端 / cutoff=打切り','値'],'rows':rows,
            'result':summary(result),'comparison':'深さ2では手A・値3、深さ1では手B・推定値4。',
            'explanation':'深さ1の値は葉まで読んだ確定利得ではなく、指定した評価関数の推定値。深く読めば常に現実の最善が保証されるという例ではない。',
            'reset':'depth=2で再実行。'}


In [ ]:
example_depth = 2
program_observation_depth = lesson_depth(example_depth)
learning_show(program_observation_depth, reset='example_depth=2に戻して再実行します。')


## 4. 境界と枝刈り

**今いる段階：** α・β→比較→省略

**予想：** 順番を変えると何が変わるか

**操作：** example_reverseをFalseからTrueへ変更し、同じセルを実行します。

**結果の目印：** 根の値3を保ち評価数が変わる

**戻す：** example_reverse=Falseに戻して再実行します。

**説明する：** 省略した葉は未計算である


### 計算の定義：`lesson_alphabeta`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_alphabeta(reverse=False):
    result = search_tree({'algorithm':'alphabeta','reverse':reverse})
    rows = []
    bounds = {}
    for event in result['trace']:
        node = event['id']
        if event['event'] in ('enter','update'):
            bounds[node] = (event['alpha'],event['beta'])
        alpha,beta = bounds.get(node,('—','—'))
        reason = ''
        if event['event']=='prune':
            reason = f'α={alpha} ≥ β={beta}。未訪問 {event["skipped"]} は根の選択を改善しない。'
        rows.append((event['event'],node,event.get('value','—'),alpha,beta,event.get('bound','—'),reason))
    full = search_tree({'reverse':reverse})
    return {'title':'αとβが出会ったら、その先を省略する','question':'省略した葉の値を計算済みと言えるか',
            'input':f'同じ木、子の順序を逆にする={reverse}',
            'columns':['処理','節点','値','α: MAXの下限','β: MINの上限','返却値の種別','省略の理由'],'rows':rows,
            'tables':[('全探索と比較',['方式','根の値','選んだ手','評価した葉数','省略'],[('minimax',full['value'],full['best_action'],full['evaluations'],full['pruned']),('αβ',result['value'],result['best_action'],result['evaluations'],result['pruned'])])],
            'result':summary(result),'comparison':'reverse=Trueでも根の値は3。評価数と省略位置は探索順に依存する。',
            'explanation':'upperは上限、lowerは下限、exactは確定値。枝刈りの返却値をすべて厳密値と読まない。-inf/+infはまだ制約のない境界。',
            'reset':'reverse=Falseで再実行。探索履歴も初めから作る。'}


In [ ]:
example_reverse = False
program_observation_alphabeta = lesson_alphabeta(example_reverse)
learning_show(program_observation_alphabeta, reset='example_reverse=Falseに戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`experiment_configs`

六つの独立変更。各ケースの基準をコピーして一項目だけ変える。


In [ ]:
def experiment_configs():
    """六つの独立変更。各ケースの基準をコピーして一項目だけ変える。"""
    cases = {}
    for name in ("leaf", "order", "depth", "heuristic", "piles", "algorithm"):
        kind = "nim" if name == "piles" else "tree"
        base = deepcopy(NIM_BASE if kind == "nim" else TREE_BASE)
        if name == "order":
            base["algorithm"] = "alphabeta"
        if name == "heuristic":
            base["depth"] = 1
        changed = deepcopy(base)
        if name == "leaf": changed["leaves"][2] = 6
        if name == "order": changed["reverse"] = True
        if name == "depth": changed["depth"] = 1
        if name == "heuristic": changed["heuristics"][1] = 5
        if name == "piles": changed["piles"] = [2, 2]
        if name == "algorithm": changed["algorithm"] = "alphabeta"
        cases[name] = {"kind": kind, "base": base, "changed": changed}
    return cases


### 計算の定義：`demo`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def demo():
    changes = {}
    for key, case in experiment_configs().items():
        run = search_nim if case["kind"] == "nim" else search_tree
        changes[key] = {"before": summary(run(case["base"])), "after": summary(run(case["changed"]))}
    return {"tree": summary(search_tree()), "alphabeta": summary(search_tree({"algorithm": "alphabeta"})),
            "cutoff": summary(search_tree({"depth": 1})), "nim": summary(search_nim()), "changes": changes}


## 基準を保持

予想：固定木のMAXはA・B・Cのどれを選ぶ？ニムは左から何個取る？想定結果は木3・A、ニム+1・[0,1]。訪問数と評価数も分けて読みます。


In [ ]:
baseline_tree_config = deepcopy(TREE_BASE)
baseline_nim_config = deepcopy(NIM_BASE)
baseline_tree = search_tree(baseline_tree_config)
baseline_nim = search_nim(baseline_nim_config)
baseline_result = demo()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))


## 完全な探索履歴

enterは訪問、evaluateは終端または打切り、updateは探索済み候補からの更新、pruneは未探索の子を省略、returnは上限・下限またはその探索条件の値です。Bの上限2を確定値と呼べない理由を説明しましょう。


In [ ]:
baseline_ab = search_tree({"algorithm": "alphabeta"})
for event in baseline_ab["trace"]:
    print(event)


## 変更：B1の利得2→6

**問い・予想：**Bが選ばれるだろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**値3・A → 値6・B。MINの値も6になる。

**説明：**相手が9でなく6を選ぶのはなぜか。

復帰はカーネル再起動→全セル実行。


In [ ]:
case_leaf = experiment_configs()["leaf"]
trial_leaf_config = deepcopy(case_leaf["changed"])
trial_leaf = search_tree(trial_leaf_config)
print("変更前", baseline_result["changes"]["leaf"]["before"])
print("変更後", summary(trial_leaf))


## 変更：αβの探索順を逆順

**問い・予想：**根値と評価数のどちらが変わるだろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**値3・Aは同じ。終端評価4→6。各子の順序も逆になる。

**説明：**Aを後に調べると省略が減る理由は何か。

復帰はカーネル再起動→全セル実行。


In [ ]:
case_order = experiment_configs()["order"]
trial_order_config = deepcopy(case_order["changed"])
trial_order = search_tree(trial_order_config)
print("変更前", baseline_result["changes"]["order"]["before"])
print("変更後", summary(trial_order))


## 変更：探索深さ2→1

**問い・予想：**深さ1でもAを選ぶだろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**確定した根値3・A → 推定値4・B。終端評価0、打切り評価3。

**説明：**深さ1の4を確定した利得と呼べるか。

復帰はカーネル再起動→全セル実行。


In [ ]:
case_depth = experiment_configs()["depth"]
trial_depth_config = deepcopy(case_depth["changed"])
trial_depth = search_tree(trial_depth_config)
print("変更前", baseline_result["changes"]["depth"]["before"])
print("変更後", summary(trial_depth))


## 変更：深さ1でAの推定値1→5

**問い・予想：**推定値だけで手が変わるだろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**推定値4・B → 推定値5・A。確定利得3とは違う。

**説明：**手が全探索と一致すれば、推定値も正しいか。

復帰はカーネル再起動→全セル実行。


In [ ]:
case_heuristic = experiment_configs()["heuristic"]
trial_heuristic_config = deepcopy(case_heuristic["changed"])
trial_heuristic = search_tree(trial_heuristic_config)
print("変更前", baseline_result["changes"]["heuristic"]["before"])
print("変更後", summary(trial_heuristic))


## 変更：ニム(3,2)→(2,2)

**問い・予想：**同じ数の山は先手に有利だろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**MAX先手の値+1 → −1。訪問86→33、終端37→14。負け局面では同点の最初の手を選ぶ。

**説明：**なぜ値−1でも合法手が返るのか。

復帰はカーネル再起動→全セル実行。


In [ ]:
case_piles = experiment_configs()["piles"]
trial_piles_config = deepcopy(case_piles["changed"])
trial_piles = search_nim(trial_piles_config)
print("変更前", baseline_result["changes"]["piles"]["before"])
print("変更後", summary(trial_piles))


## 変更：ミニマックス→αβ

**問い・予想：**省略によって根値が変わるだろうか。

手順：対応ケースの基準をdeepcopyし、一項目だけ変更した入力で実行。

**想定結果：**値3・Aは同じ。訪問10→8、終端評価6→4。

**説明：**未探索のB2・C2の確定値を表示できない理由は何か。

復帰はカーネル再起動→全セル実行。


In [ ]:
case_algorithm = experiment_configs()["algorithm"]
trial_algorithm_config = deepcopy(case_algorithm["changed"])
trial_algorithm = search_tree(trial_algorithm_config)
print("変更前", baseline_result["changes"]["algorithm"]["before"])
print("変更後", summary(trial_algorithm))


## 基準の保持と復帰

下の検査が成功すれば、基準と6実験が混ざっていません。ValueErrorは入力の範囲と型を、NameErrorは実行順とカーネルを確認します。定義セルを消してしまった場合は保存した元ファイルへ戻します。自分の変更は新しいセルで基準のコピーを使います。


In [ ]:
assert baseline_tree_config == TREE_BASE
assert baseline_nim_config == NIM_BASE
assert baseline_tree == search_tree(TREE_BASE)
assert baseline_nim == search_nim(NIM_BASE)
assert baseline_result == demo()
for key, case in experiment_configs().items():
    run = search_nim if case["kind"] == "nim" else search_tree
    assert globals()["trial_" + key + "_config"] == case["changed"]
    assert globals()["trial_" + key] == run(case["changed"])
print("基準入力・結果を保持しました")


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python game_search.py --explain all`で同じ計算表を表示します。各テーマは`--explain moves`のように指定できます。従来のコマンドと固定期待値も維持しています。
